<a href="https://colab.research.google.com/github/idialloaka-ai/DAILYCHALLENGE/blob/master/Mini_projet_week9_day5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# MINI-PROJET : MCP + Agents IA intégrés dans Gemini
# Version finale – compatible avec langchain-mcp-adapters 0.2.1

# 1. Installation des dépendances
%pip install -qU \
  "langchain>=0.3" \
  "langgraph>=0.2" \
  "langchain-google-genai>=2.0" \
  "google-genai>=1.0" \
  "langchain-mcp-adapters==0.2.1" \
  "nest_asyncio" \
  "fastmcp>=2.0.0"

%pip install -qU mcp-server-git 2>/dev/null || echo "  mcp-server-git non installé via pip, sera lancé via npx"

# 2. Configuration de l'environnement (clé API + asyncio)
import os
import nest_asyncio
nest_asyncio.apply()

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = input(" Entrez votre clé API Google : ")

print(" Clé API configurée")

# 3. Vérification de Node.js / npm
import subprocess
try:
    subprocess.run(["node", "--version"], check=True, capture_output=True)
    print(" Node.js détecté")
except:
    !apt-get -qq update && apt-get -qq install -y nodejs npm
    print(" Node.js installé")

try:
    subprocess.run(["npx", "--version"], check=True, capture_output=True)
    print(" npx détecté")
except:
    !npm install -g npx

# 4. Création de l'espace de travail (pour les tests)
WORKDIR = "/content/workspace"
!mkdir -p {WORKDIR}
!echo "Bonjour, ceci est un fichier test." > {WORKDIR}/test.txt
!echo "Ligne 2 : MCP est génial." >> {WORKDIR}/test.txt
!echo "Ligne 3 : Gemini orchestre tout." >> {WORKDIR}/test.txt
!cd {WORKDIR} && git init && git config user.email "test@example.com" && git config user.name "Test User" && git add test.txt && git commit -m "Initial commit" 2>/dev/null
print(f" Espace de travail prêt : {WORKDIR}")

# 5. Création du serveur MCP personnalisé (FastMCP)
from pathlib import Path
import textwrap

SERVER_PATH = Path("/content/custom_mcp_server.py")
SERVER_PATH.write_text(textwrap.dedent("""
    from fastmcp import FastMCP
    from typing import Dict, List

    mcp = FastMCP(name="custom_ops")

    @mcp.tool
    def ping() -> str:
        \"\"\"Health check tool.\"\"\"
        return "pong"

    @mcp.tool
    def summarize_lines(lines: List[str]) -> Dict[str, int]:
        \"\"\"Compte les lignes totales et non vides.\"\"\"
        total = len(lines)
        nonempty = sum(1 for l in lines if l.strip())
        return {"total_lines": total, "nonempty_lines": nonempty}

    @mcp.tool
    def format_uppercase(text: str) -> str:
        \"\"\"Convertit un texte en majuscules.\"\"\"
        return text.upper()

    if __name__ == "__main__":
        mcp.run(transport="stdio")
"""), encoding="utf-8")

print(f" Serveur MCP personnalisé créé : {SERVER_PATH}")

# 6. Définition des connexions MCP
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_connections = {
    "filesystem": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", WORKDIR],
    },
    "git": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-git", WORKDIR],
    },
    "custom_ops": {
        "transport": "stdio",
        "command": "python",
        "args": [str(SERVER_PATH)],
    },
}

print(" Connexions MCP définies")

# 7. Fonction principale asynchrone (sans context manager)
import asyncio
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent

async def main():
    # Création du client (pas de async with)
    client = MultiServerMCPClient(mcp_connections, tool_name_prefix=True)
    try:
        tools = await client.get_tools()
        print(f"\n {len(tools)} outils chargés :")
        for t in tools:
            print(f"  - {t.name}")

        llm = ChatGoogleGenerativeAI(
            model="gemini-1.5-flash",
            temperature=0,
            google_api_key=os.environ["GOOGLE_API_KEY"]
        )

        agent = create_react_agent(llm, tools)

        # Requêtes de test
        queries = [
            "Liste tous les fichiers dans /content/workspace.",
            "Donne-moi le dernier commit du dépôt Git dans /content/workspace.",
            "Lis le fichier /content/workspace/test.txt, puis utilise l'outil summarize_lines pour compter les lignes.",
            "Récupère le contenu de test.txt, mets-le en majuscules avec l'outil custom_ops_format_uppercase, puis affiche le résultat."
        ]

        for q in queries:
            print("\n" + "="*60)
            print(f" QUESTION : {q}")
            print("="*60)
            result = await agent.ainvoke({"messages": [("user", q)]})
            print("\n RÉPONSE :")
            print(result["messages"][-1].content)
            print("="*60)

    finally:
        # Fermeture propre des sous-processus MCP
        await client.close()
        print("\n Connexions MCP fermées.")

# 8. Lancement (Colab supporte await en top-level)
await main()

RecursionError: maximum recursion depth exceeded